## Cell 1 — Install dependencies

Run once per session. Takes ~3 minutes.


In [1]:
import subprocess, sys, os

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"STDERR: {result.stderr[-2000:]}")
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout

# Boltz-2
run("pip install boltz --quiet")

# USalign — compile from source (pylelab/USalign)
if not os.path.exists("/content/USalign/USalign"):
    run("git clone --quiet https://github.com/pylelab/USalign.git /content/USalign")
    run("cd /content/USalign && g++ -O3 -ffast-math -lm -o USalign USalign.cpp")
USALIGN_BIN = "/content/USalign/USalign"

# DSSP (optional — beta-sheet orientation filter degrades gracefully without it)
run("pip install biopython --quiet")
try:
    run("apt-get install -y -q dssp")
except Exception:
    pass
run("pip install cuequivariance-torch cuequivariance-ops-torch-cu12 --quiet")

# FreeSASA and gemmi
run("pip install freesasa gemmi --quiet")

print("✓ All dependencies installed")
print(f"✓ USalign binary: {USALIGN_BIN}")
# Download trunk scoring script
run("wget -q -O /content/trunk_score.py https://raw.githubusercontent.com/GuanrongHuang/Scramble/main/trunk_score.py")
print("✓ trunk_score.py ready")


✓ All dependencies installed
✓ USalign binary: /content/USalign/USalign
✓ trunk_score.py ready


In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Helper functions
# Run once per session. All pipeline stages depend on these.
#
# Organisation:
#   1. Imports and constants
#   2. IPSAE and ranking formula   (MOSAIC-equivalent)
#   3. Structure loading           (gemmi, PDB + CIF)
#   4. Secondary structure         (DSSP)
#   5. Solvent accessibility       (FreeSASA)
#   6. Geometric utilities         (RMSD, beta-sheet orientation)
#   7. USalign wrappers            (complex alignment, single-chain RMSD)
#   8. Boltz-2                     (run, score, helpers)
#   9. CSV loading
# ═══════════════════════════════════════════════════════════════════════════════

import glob, io, json, math, itertools, time
import numpy as np
from pathlib import Path


# ── 1. Constants ───────────────────────────────────────────────────────────────

# Boltz-2 / AlphaFold PAE bin structure
# 64 bins, breaks at linspace(0, 31, 63), centres at [0.25, 0.75, ..., 31.75]
_PAE_BREAKS      = np.linspace(0.0, 31.0, 63, dtype=np.float32)
_PAE_BIN_CENTERS = np.concatenate(
    [_PAE_BREAKS + 0.25, [_PAE_BREAKS[-1] + 0.75]]
).astype(np.float32)   # shape (64,)

THREE_TO_ONE = {
    "ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C",
    "GLN":"Q","GLU":"E","GLY":"G","HIS":"H","ILE":"I",
    "LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P",
    "SER":"S","THR":"T","TRP":"W","TYR":"Y","VAL":"V",
}


# ── 2. IPSAE and ranking formula ───────────────────────────────────────────────
#
# MOSAIC source (src/mosaic/losses/structure_prediction.py):
#
#   probs = softmax(logits, axis=-1)            # [N, N, Bins]
#   pae   = sum(probs * bin_centers, axis=-1)   # [N, N]
#
#   pair_mask  = asym_id[:,None] != asym_id[None,:]
#   pair_mask *= pae < pae_cutoff
#   n_residues = sum(pair_mask, axis=-1, keepdims=True)
#   d0 = 1.24 * (clip(n_residues, min=27) - 15)^(1/3) - 1.8
#
#   tm_per_bin        = 1 / (1 + bin_centers^2 / d0^2)
#   predicted_tm_term = sum(probs * tm_per_bin, axis=-1)
#   normed            = pair_mask / (1e-8 + n_residues)
#   per_alignment     = sum(predicted_tm_term * normed, axis=-1)  # [N]
#
#   BinderTargetIPSAE = max(per_alignment[:binder_len])
#   TargetBinderIPSAE = max(per_alignment[binder_len:])
#
# MOSAIC uses raw PAE logits from the model forward pass.
# Native Boltz-2 CLI outputs scalar PAE (expected value post-softmax).
# We reconstruct an approximate probability distribution over the 64 PAE bins
# using a Gaussian kernel centred at the scalar PAE (sigma = 0.5 A = 1 bin).
# Diagnostic testing showed this is numerically equivalent to the Dirac delta
# approximation (delta < 0.0002). The primary cause of score mismatch vs MOSAIC
# is MSA usage: MOSAIC uses an MSA for the target; running with msa: empty
# produces significantly different predictions. Set TARGET_MSA to a pre-generated
# MSA path (see Cell 4b) or "server" to match MOSAIC scoring.
# ──────────────────────────────────────────────────────────────────────────────

def compute_ipsae(pae: np.ndarray, binder_len: int,
                  pae_cutoff: float = 10.0) -> tuple:
    """
    Compute BinderTargetIPSAE and TargetBinderIPSAE from scalar PAE matrix,
    using the exact MOSAIC formula structure with a reconstructed bin
    probability distribution.

    Args:
        pae:        float32 [N, N]  scalar PAE from pae_*.npz
        binder_len: number of binder residues (chain A, indices 0..binder_len-1)
        pae_cutoff: cross-chain contact gate in Angstrom (MOSAIC default = 10.0)

    Returns:
        (ipsae_bt, ipsae_tb)  higher = better interface confidence
    """
    pae = np.asarray(pae, dtype=np.float32)
    N   = pae.shape[0]

    # Reconstruct approximate probability distribution over bins
    sigma  = 0.5   # one bin width in Angstrom
    diff   = pae[:, :, None] - _PAE_BIN_CENTERS[None, None, :]  # [N, N, 64]
    lp     = -0.5 * (diff / sigma) ** 2
    probs  = np.exp(lp - lp.max(axis=-1, keepdims=True))
    probs /= probs.sum(axis=-1, keepdims=True)                   # [N, N, 64]

    # Expected PAE from reconstructed distribution
    pae_r = (probs * _PAE_BIN_CENTERS).sum(axis=-1)              # [N, N]

    # Cross-chain mask + PAE cutoff gate
    asym_id     = np.concatenate([np.zeros(binder_len, dtype=np.float32),
                                   np.ones(N - binder_len, dtype=np.float32)])
    cross_chain = asym_id[:, None] != asym_id[None, :]
    pair_mask   = cross_chain & (pae_r < pae_cutoff)
    pair_mask_f = pair_mask.astype(np.float32)                   # [N, N]

    # Adaptive d0 per residue (Dunbrack 2025, eqn 15)
    n_residues = pair_mask_f.sum(axis=1, keepdims=True)          # [N, 1]
    d0         = 1.24 * (np.clip(n_residues, 27, None) - 15) ** (1.0/3) - 1.8

    # Distribution-weighted TM score per bin pair (MOSAIC form)
    tm_per_bin   = 1.0 / (1.0 + (_PAE_BIN_CENTERS ** 2) / (d0 ** 2))
    predicted_tm = (probs * tm_per_bin).sum(axis=-1)             # [N, N]

    # Normalised mask and per-alignment score
    normed        = pair_mask_f / (1e-8 + n_residues)
    per_alignment = (predicted_tm * normed).sum(axis=1)          # [N]

    ipsae_bt = float(per_alignment[:binder_len].max()) if binder_len > 0 else 0.0
    ipsae_tb = float(per_alignment[binder_len:].max()) if (N - binder_len) > 0 else 0.0
    return ipsae_bt, ipsae_tb


def ranking_score(iptm: float, ipsae_bt: float, ipsae_tb: float) -> float:
    """
    MOSAIC ranking formula (exact):
        ranking_score = -(1.0 * iptm + 0.5 * ipsae_bt + 0.5 * ipsae_tb)
    Lower (more negative) = better predicted binder.
    """
    return -(1.0 * iptm + 0.5 * ipsae_bt + 0.5 * ipsae_tb)


# ── 3. Structure loading ───────────────────────────────────────────────────────

def get_chain_residues(structure_path, chain_id):
    """
    Return ordered list of residue dicts for a chain.
    Accepts both PDB and CIF via gemmi.read_structure().
    Each dict: {ca, cb, all_atoms, aa, seqnum}
    """
    import gemmi
    st = gemmi.read_structure(str(structure_path))
    result = []
    for model in st:
        for chain in model:
            if chain.name != chain_id:
                continue
            for res in chain:
                aa1   = THREE_TO_ONE.get(res.name, "X")
                atoms = {}
                ca = cb = None
                for atom in res:
                    xyz = np.array([atom.pos.x, atom.pos.y, atom.pos.z])
                    atoms[atom.name] = xyz
                    if atom.name == "CA": ca = xyz
                    if atom.name == "CB": cb = xyz
                if ca is None:
                    continue
                result.append({"ca": ca, "cb": cb, "all_atoms": atoms,
                                "aa": aa1, "seqnum": res.seqid.num})
        break  # first model only
    return result


def get_chain_sequence(structure_path, chain_id):
    return "".join(r["aa"] for r in get_chain_residues(structure_path, chain_id))


def auto_detect_target_chain(structure_path, binder_chain):
    """Return the first chain ID that is not the binder chain."""
    import gemmi
    st = gemmi.read_structure(str(structure_path))
    for model in st:
        for chain in model:
            if chain.name != binder_chain:
                return chain.name
    return None


# ── 4. Secondary structure (DSSP) ─────────────────────────────────────────────

def get_dssp(structure_path, chain_id):
    """
    Returns {0-based-idx: ss_code} for the chain.
    CIF files are converted to a temp PDB before calling mkdssp.
    Returns {} gracefully if DSSP/biopython is unavailable.
    """
    try:
        from Bio.PDB import PDBParser, DSSP as BioDSSP
    except ImportError:
        print("  [warn] biopython unavailable — beta-sheet filter skipped")
        return {}
    import tempfile, gemmi

    suffix  = os.path.splitext(str(structure_path))[1].lower()
    tmp_pdb = None
    try:
        if suffix in (".cif", ".mmcif"):
            st = gemmi.read_structure(str(structure_path))
            with tempfile.NamedTemporaryFile(suffix=".pdb", delete=False) as tmp:
                tmp_pdb = tmp.name
            st.write_pdb(tmp_pdb)
            pdb_for_dssp = tmp_pdb
        else:
            pdb_for_dssp = str(structure_path)

        parser    = PDBParser(QUIET=True)
        structure = parser.get_structure("s", pdb_for_dssp)
        dssp      = BioDSSP(structure[0], pdb_for_dssp, dssp="mkdssp")
    except Exception as e:
        print(f"  [warn] DSSP failed ({e}) — beta-sheet filter skipped")
        return {}
    finally:
        if tmp_pdb:
            try: os.unlink(tmp_pdb)
            except OSError: pass

    result = {}
    idx = 0
    for key in dssp.property_keys:
        ch, _ = key
        if ch != chain_id:
            continue
        ss = dssp[key][2]
        result[idx] = ss if ss and ss.strip() else "C"
        idx += 1
    return result


# ── 5. Solvent accessibility (FreeSASA) ───────────────────────────────────────

def get_sasa(structure_path, chain_id):
    """Returns {0-based-idx: sasa_float}. Returns {} if freesasa unavailable."""
    import tempfile, gemmi as _gemmi_sasa

    suffix  = os.path.splitext(str(structure_path))[1].lower()
    tmp_pdb = None
    try:
        if suffix in (".cif", ".mmcif"):
            st = _gemmi_sasa.read_structure(str(structure_path))
            with tempfile.NamedTemporaryFile(suffix=".pdb", delete=False) as tmp:
                tmp_pdb = tmp.name
            st.write_pdb(tmp_pdb)
            pdb_for_sasa = tmp_pdb
        else:
            pdb_for_sasa = str(structure_path)

        import freesasa
        structure = freesasa.Structure(pdb_for_sasa)
        result    = freesasa.calc(structure)
        classes   = freesasa.classifyResults(result, structure)

        # Sum atom-level SASA per residue for the target chain
        # classifyResults returns {'polar': [...], 'apolar': [...]}
        # We sum polar + apolar per residue index
        residue_totals = {}
        for i in range(structure.nAtoms()):
            if structure.chainLabel(i) != chain_id:
                continue
            resnum = structure.residueNumber(i)
            area   = result.atomArea(i)
            if resnum not in residue_totals:
                residue_totals[resnum] = 0.0
            residue_totals[resnum] += area

        # Convert to 0-based index map in residue order
        sasa_map = {}
        for idx, resnum in enumerate(sorted(residue_totals.keys())):
            sasa_map[idx] = residue_totals[resnum]
        return sasa_map

    except Exception as e:
        print(f"  [warn] FreeSASA failed ({e}) — SASA filter skipped")
        return {}
    finally:
        if tmp_pdb:
            try: os.unlink(tmp_pdb)
            except OSError: pass

# ── 6. Geometric utilities ─────────────────────────────────────────────────────

def per_residue_rmsd(d_res, k_res):
    """All-atom RMSD between two residue dicts over shared atom names."""
    common = set(d_res["all_atoms"]) & set(k_res["all_atoms"])
    if not common:
        return float("inf")
    diffs = np.array([d_res["all_atoms"][a] - k_res["all_atoms"][a]
                      for a in common])
    return float(np.sqrt(np.mean(np.sum(diffs**2, axis=1))))


def sheet_normal(ca_prev, ca_curr, ca_next):
    """Outward normal of a beta-sheet plane from three consecutive Ca positions."""
    v1   = ca_next - ca_curr
    v2   = ca_prev - ca_curr
    n    = np.cross(v1, v2)
    norm = np.linalg.norm(n)
    return n / norm if norm > 1e-6 else np.zeros(3)


def cb_dot(ca, cb, normal):
    """Dot product of the Ca->Cb unit vector with a sheet normal."""
    if cb is None:
        return float("nan")
    v    = cb - ca
    norm = np.linalg.norm(v)
    if norm < 1e-6:
        return float("nan")
    return float(np.dot(v / norm, normal))


# ── 7. USalign wrappers ────────────────────────────────────────────────────────

def _parse_aln_file(aln_path):
    """Parse a USalign .aln file into a list of (header, sequence) tuples."""
    entries, current_header, current_seq = [], None, []
    with open(aln_path) as f:
        for line in f:
            line = line.rstrip()
            if line.startswith(">"):
                if current_header is not None:
                    entries.append((current_header, "".join(current_seq)))
                current_header = line[1:].strip()
                current_seq    = []
            else:
                current_seq.append(line.strip())
    if current_header is not None:
        entries.append((current_header, "".join(current_seq)))
    return entries


def run_usalign_complex(query_cif, reference_pdb, query_binder_chain,
                        ref_binder_chain, out_prefix):
    """
    Align predicted complex vs known complex using USalign multimer mode (-mm 1).
    Both chains guide the superposition — the target anchors binding orientation.
    Returns (tmscore, rmsd, residue_map {query_binder_0based: ref_binder_0based}).
    """
    cmd = [
        USALIGN_BIN,
        str(query_cif), str(reference_pdb),
        "-mm",     "1",
        "-ter",    "0",
        "-outfmt", "2",
        "-o",      str(out_prefix),
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
    if proc.returncode != 0:
        raise RuntimeError(f"USalign (complex) failed:\n{proc.stderr}")

    tmscore = rmsd = 0.0
    for line in proc.stdout.splitlines():
        line = line.strip()
        if line.startswith("#") or not line:
            continue
        parts = line.split()
        if len(parts) >= 3:
            try:
                tmscore = float(parts[0])
                rmsd    = float(parts[2])
                break
            except ValueError:
                continue

    residue_map = {}
    aln_files   = list(Path(out_prefix).parent.glob(
                        Path(out_prefix).name + "*.aln"))
    if not aln_files:
        return tmscore, rmsd, residue_map

    entries = _parse_aln_file(aln_files[0])

    query_binder_seq = ref_binder_seq = None
    for i in range(0, len(entries) - 1, 2):
        hdr_q, seq_q = entries[i]
        hdr_r, seq_r = entries[i + 1]
        if (hdr_q.strip()[-1] == query_binder_chain and
                hdr_r.strip()[-1] == ref_binder_chain):
            query_binder_seq = seq_q
            ref_binder_seq   = seq_r
            break
    if query_binder_seq is None and len(entries) >= 2:
        query_binder_seq = entries[0][1]
        ref_binder_seq   = entries[1][1]

    if (query_binder_seq is not None and ref_binder_seq is not None and
            len(query_binder_seq) == len(ref_binder_seq)):
        qi = ki = 0
        for qc, kc in zip(query_binder_seq, ref_binder_seq):
            if qc != "-" and kc != "-":
                residue_map[qi] = ki
            if qc != "-": qi += 1
            if kc != "-": ki += 1

    return tmscore, rmsd, residue_map


def run_usalign_single(query, reference, ref_chain, out_prefix):
    """
    Single-chain alignment for the post-prediction RMSD gate (Stage 7).
    Returns (tmscore, rmsd).
    """
    cmd = [
        USALIGN_BIN,
        str(query), str(reference),
        "-chain2", ref_chain,
        "-ter",    "0",
        "-outfmt", "2",
        "-o",      str(out_prefix),
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    if proc.returncode != 0:
        raise RuntimeError(f"USalign (single) failed:\n{proc.stderr}")

    tmscore = rmsd = 0.0
    for line in proc.stdout.splitlines():
        line = line.strip()
        if line.startswith("#") or not line:
            continue
        parts = line.split()
        if len(parts) >= 3:
            try:
                tmscore = float(parts[0])
                rmsd    = float(parts[2])
                break
            except ValueError:
                continue
    return tmscore, rmsd


# ── 8. Boltz-2 ────────────────────────────────────────────────────────────────

def _boltz_target_msa_line():
    """
    Return the YAML msa line for the target chain.

    TARGET_MSA options (set in Cell 2):
      "empty"      — no MSA (fast, but degrades accuracy vs MOSAIC)
      "server"     — fetch via mmseqs2 server at prediction time
      "<path>"     — pre-generated MSA file (.a3m or .csv, from Cell 4b)
    """
    if TARGET_MSA == "server":
        return ""                          # --use_msa_server flag handles it
    elif TARGET_MSA == "empty":
        return "\n      msa: empty"
    else:
        assert os.path.exists(TARGET_MSA), (
            f"TARGET_MSA path not found: {TARGET_MSA}\n"
            "Run the MSA generation cell (Cell 4b) first, "
            "or set TARGET_MSA = 'server' in Cell 2."
        )
        return f"\n      msa: {TARGET_MSA}"


def run_boltz(name, binder_seq, target_seq, work_dir,
              recycling_steps=None, diffusion_samples=None, sampling_steps=None):
    """
    Write YAML input and run boltz predict.

    Binder = chain A, msa: empty (no evolutionary history for designed seqs).
    Target = chain B, msa: controlled by TARGET_MSA (Cell 2).

    IMPORTANT: For MOSAIC-equivalent scoring the target must have an MSA.
    Set TARGET_MSA to the path from the MSA generation cell (Cell 4b),
    or set TARGET_MSA = "server" to fetch at prediction time (~30s overhead).
    Running with TARGET_MSA = "empty" produces significantly lower and
    less reliable scores than MOSAIC reports.

    Returns the prediction output directory path.
    """
    recycling_steps   = recycling_steps   or RECYCLING_STEPS
    diffusion_samples = diffusion_samples or DIFFUSION_SAMPLES
    sampling_steps    = sampling_steps    or SAMPLING_STEPS

    os.makedirs(work_dir, exist_ok=True)
    safe      = "".join(c if c.isalnum() or c == "_" else "_" for c in name)[:40]
    yaml_path = os.path.join(work_dir, f"{safe}.yaml")

    with open(yaml_path, "w") as f:
        f.write(f"""version: 1
sequences:
  - protein:
      id: [A]
      sequence: {binder_seq}
      msa: empty
  - protein:
      id: [B]
      sequence: {target_seq}{_boltz_target_msa_line()}
""")

    import shutil
    stale_dir = os.path.join(work_dir, f"boltz_results_{safe}")
    if os.path.exists(stale_dir):
        shutil.rmtree(stale_dir)

    cmd = [
        "boltz", "predict", yaml_path,
        "--out_dir",           work_dir,
        "--model",             "boltz2",
        "--recycling_steps",   str(recycling_steps),
        "--sampling_steps",    str(sampling_steps),
        "--diffusion_samples", str(diffusion_samples),
        "--write_full_pae",
        "--write_embeddings",
        "--accelerator",       "gpu",
        "--override",
    ]
    if TARGET_MSA == "server":
        cmd += ["--use_msa_server"]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        raise RuntimeError(
            f"boltz predict failed for {name}:\n"
            f"STDOUT: {result.stdout[-1000:]}\nSTDERR: {result.stderr[-1000:]}"
        )


    # Boltz-2 writes to: <work_dir>/boltz_results_<yaml_stem>/predictions/<yaml_stem>/
    pred_dir = os.path.join(work_dir, f"boltz_results_{safe}", "predictions", safe)
    if not os.path.isdir(pred_dir):
        for root, dirs, files in os.walk(work_dir):
            if any(f.endswith(".cif") for f in files):
                pred_dir = root
                break
    if not os.path.isdir(pred_dir):
        raise RuntimeError(
            f"Could not find Boltz-2 output directory for {name}.\n"
            f"Contents of {work_dir}:\n"
            + "\n".join(f"  {dp}/{f}"
                        for dp, dn, fn in os.walk(work_dir) for f in fn)
        )
    return pred_dir


def score_prediction(pred_dir, binder_len, target_len):
    """
    Average metrics across all diffusion samples and return a scoring dict.
    Uses protein_iptm (protein-protein interface ipTM) to match MOSAIC's
    IPTMLoss, which ignores ligand-chain pairs.
    """
    conf_files = sorted(glob.glob(os.path.join(pred_dir, "confidence_*.json")))
    pae_files  = sorted(glob.glob(os.path.join(pred_dir, "pae_*.npz")))
    if not conf_files:
        raise FileNotFoundError(f"No confidence_*.json in {pred_dir}")
    if not pae_files:
        raise FileNotFoundError(f"No pae_*.npz in {pred_dir}")

    n = min(len(conf_files), len(pae_files))
    iptms, plddts, ptms, bt_list, tb_list = [], [], [], [], []

    for cf, pf in zip(conf_files[:n], pae_files[:n]):
        with open(cf) as f:
            conf = json.load(f)

        iptm  = conf.get("protein_iptm", conf.get("iptm", 0.0))
        plddt = conf.get("complex_plddt", conf.get("plddt", 0.0))
        ptm   = conf.get("ptm", 0.0)

        data  = np.load(pf)
        pae   = data["pae"] if "pae" in data else list(data.values())[0]
        if pae.ndim == 3: pae = pae[0]
        pae   = pae.astype(np.float32)

        expected = binder_len + target_len
        if pae.shape != (expected, expected):
            raise ValueError(
                f"PAE shape {pae.shape} != expected ({expected},{expected})"
            )

        bt, tb = compute_ipsae(pae, binder_len)
        iptms.append(iptm); plddts.append(plddt); ptms.append(ptm)
        bt_list.append(bt); tb_list.append(tb)

    iptm_m     = float(np.mean(iptms))
    plddt_m    = float(np.mean(plddts))
    ptm_m      = float(np.mean(ptms))
    ipsae_bt_m = float(np.mean(bt_list))
    ipsae_tb_m = float(np.mean(tb_list))

    return {
        "ranking_score": ranking_score(iptm_m, ipsae_bt_m, ipsae_tb_m),
        "iptm":     iptm_m,
        "ptm":      ptm_m,
        "plddt":    plddt_m,
        "ipsae_bt": ipsae_bt_m,
        "ipsae_tb": ipsae_tb_m,
    }


def get_best_cif(pred_dir):
    """Return model_0 (highest-confidence) predicted .cif structure path."""
    cifs = sorted(glob.glob(os.path.join(pred_dir, "*.cif")))
    if not cifs: return None
    for cif in cifs:
        if "model_0" in cif: return cif
    return cifs[0]


def get_cross_pae_per_binder_residue(pred_dir, binder_len, target_len):
    """
    For each binder residue i, compute mean(pae[i, binder_len:]) —
    mean PAE across all target-chain columns in row i.
    Lower = model is more confident about where residue i sits relative
    to the target. Used as Gate 1 in Stage 4 residue flagging.
    Falls back to [inf]*binder_len if PAE files are unavailable.
    """
    pae_files = sorted(glob.glob(os.path.join(pred_dir, "pae_*.npz")))
    if not pae_files:
        return [float("inf")] * binder_len

    expected    = binder_len + target_len
    accumulated = np.zeros(binder_len, dtype=np.float64)
    n_valid     = 0

    for pf in pae_files:
        data = np.load(pf)
        pae  = data["pae"] if "pae" in data else list(data.values())[0]
        if pae.ndim == 3: pae = pae[0]
        pae = pae.astype(np.float32)
        if pae.shape != (expected, expected):
            continue
        accumulated += pae[:binder_len, binder_len:].mean(axis=1)
        n_valid += 1

    if n_valid == 0:
        return [float("inf")] * binder_len
    return (accumulated / n_valid).tolist()


# ── 9. CSV loading ─────────────────────────────────────────────────────────────

def load_designs(csv_path):
    """
    Load designs CSV. Handles both headered and headerless files.
    Headerless: first valid all-AA column is the sequence, auto-names designs.
    Headered: tolerates any capitalisation of name and sequence columns.
    """
    with open(csv_path, newline="") as f:
        raw = f.read()

    lines        = [l.strip() for l in raw.splitlines() if l.strip()]
    first_fields = lines[0].split(",") if lines else []
    has_header   = (
        len(first_fields) >= 1 and
        not all(c in VALID_AA
                for c in first_fields[0].upper().replace("-","").replace("_",""))
    )

    rows = []
    f2   = io.StringIO(raw)

    if has_header:
        reader     = csv.DictReader(f2)
        raw_fields = reader.fieldnames or []
        field_map  = {f.strip().lower(): f for f in raw_fields}
        seq_field  = field_map.get("sequence") or field_map.get("seq")
        name_field = field_map.get("name") or field_map.get("id")
        if seq_field is None:
            raise ValueError(
                f"CSV has a header but no 'sequence' column. Found: {raw_fields}"
            )
        for i, row in enumerate(reader):
            seq  = row[seq_field].strip().upper()
            name = (row[name_field].strip()
                    if name_field and row.get(name_field)
                    else f"binder_{i+1}")
            bad  = set(seq) - VALID_AA
            if bad:
                print(f"  [skip] {name}: invalid characters {bad}")
                continue
            rows.append({"name": name, "sequence": seq})
    else:
        reader = csv.reader(f2)
        for i, cols in enumerate(reader):
            if not cols:
                continue
            seq_col = None
            for col in cols:
                candidate = col.strip().upper()
                if candidate and all(c in VALID_AA for c in candidate):
                    seq_col = candidate
                    break
            if seq_col is None:
                continue
            name = f"binder_{i+1}"
            for col in cols:
                if col.strip() == seq_col:
                    continue
                try:
                    float(col)
                except ValueError:
                    name = col.strip()
                    break
            rows.append({"name": name, "sequence": seq_col})

    print(f"  Detected {'header' if has_header else 'no header'} in CSV")
    return rows


# ── Confirm all functions loaded ───────────────────────────────────────────────
_expected = [
    "compute_ipsae", "ranking_score",
    "get_chain_residues", "get_chain_sequence", "auto_detect_target_chain",
    "get_dssp", "get_sasa",
    "per_residue_rmsd", "sheet_normal", "cb_dot",
    "_parse_aln_file", "run_usalign_complex", "run_usalign_single",
    "_boltz_target_msa_line", "run_boltz", "score_prediction", "get_best_cif",
    "get_cross_pae_per_binder_residue", "load_designs",
]
for _fn in _expected:
    print(f"  {'✓' if _fn in dir() else '✗ MISSING'}  {_fn}")
print("✓ Cell 5 loaded")

  ✓  compute_ipsae
  ✓  ranking_score
  ✓  get_chain_residues
  ✓  get_chain_sequence
  ✓  auto_detect_target_chain
  ✓  get_dssp
  ✓  get_sasa
  ✓  per_residue_rmsd
  ✓  sheet_normal
  ✓  cb_dot
  ✓  _parse_aln_file
  ✓  run_usalign_complex
  ✓  run_usalign_single
  ✓  _boltz_target_msa_line
  ✓  run_boltz
  ✓  score_prediction
  ✓  get_best_cif
  ✓  get_cross_pae_per_binder_residue
  ✓  load_designs
✓ Cell 5 loaded


## Cell 2 — Configuration

Edit these parameters before running the pipeline.
Boltz-2 settings are taken directly from the MOSAIC/Escalante
inference defaults and should not normally need to be changed.


In [3]:
# ── Boltz-2 settings (MOSAIC Escalante inference defaults) ────────────────────
RECYCLING_STEPS   = 3
DIFFUSION_SAMPLES = 6
SAMPLING_STEPS    = 200   # reduce to 50 for speed with minimal accuracy loss

# ── Geometric filter thresholds ───────────────────────────────────────────────
RMSD_THRESHOLD  = 3.0   # Å — per-residue all-atom RMSD gate for flagging
MAX_CROSS_PAE   = 10.0   # Å — maximum mean cross-chain PAE at a flagged position
                         # (lower = more confident interface placement; 5-10 A typical)
MIN_SASA        = 5.0  # Å² — minimum FreeSASA for a flagged position
DOT_PRODUCT_MIN = 0.1   # minimum Cβ-normal dot product (strand residues only)
EXCLUDE_AA      = {"P"} # amino acids to never substitute in

# ── Variant generation ────────────────────────────────────────────────────────
MAX_SUBSTITUTIONS = 3   # max simultaneous substitutions per variant
MAX_VARIANTS      = 9  # max variants per input design

# ── Screening filters ─────────────────────────────────────────────────────────
MIN_IPSAE_SCREEN    = 0.6  # minimum ipsae_bt or ipsae_tb to pass initial screen
MAX_RMSD_SCREEN     = 3.0  # Å — max global RMSD vs known contact fragment
RMSD_GATE_VS_PARENT = 1.5  # Å — max RMSD of variant vs original prediction

# ── MSA settings ──────────────────────────────────────────────────────────────
# Binder MSA is always empty (designed sequences have no evolutionary history).
# TARGET_MSA: "empty" (fast, matches MOSAIC default) | "server" (better accuracy)
TARGET_MSA = "server"
print(f"✓ TARGET_MSA set to: {TARGET_MSA}")

# ── Output ────────────────────────────────────────────────────────────────────
TOP_N       = 60
OUTPUT_CSV  = "binders_optimised_ranked.csv"
OUT_DIR     = "/content/pipeline_output"

print("✓ Configuration set")


✓ TARGET_MSA set to: server
✓ Configuration set


## Cell 3 — ✏️ Enter your target sequence and known binder settings

**Fill in all fields here before running Cell 4.**

### Input PDB/CIF format
The known binder file you upload in Cell 4 must be **pre-trimmed**
so that the binder chain contains **only the contact residues** —
the interface residues that form the binding epitope. Every residue
present in that chain will be treated as a contact residue and used
as a substitution anchor. Do not include the full binder sequence.

The file may contain the target chain as well (or just the trimmed
binder chain alone). Both PDB and CIF formats are accepted.

### TARGET_SEQUENCE
Paste the **full** amino acid sequence of your target protein.
If left empty the pipeline will attempt to auto-extract it from
the non-binder chain of the uploaded PDB, but providing it
explicitly is strongly recommended.

### KNOWN_BINDER_CHAIN
The chain ID of the trimmed known binder in the uploaded file.


In [4]:
# ┌─────────────────────────────────────────────────────────────────────────┐
# │  TARGET SEQUENCE  (full amino acid sequence of the target protein)      │
# └─────────────────────────────────────────────────────────────────────────┘
TARGET_SEQUENCE = "MAAAMDVDTPSGTNSGAGKKRFEVKKWNAVALWAWDIVVDNCAICRNHIMDLCIECQANQASATSEECTVAWGVCNHAFHFHCISRWLKTRQVCPLDNREWEFQKYGH"  # e.g. "MKTAYIAKQRQISFVKSHFSRQ..."

# ┌─────────────────────────────────────────────────────────────────────────┐
# │  KNOWN BINDER CHAIN ID  (chain containing only the contact residues)    │
# └─────────────────────────────────────────────────────────────────────────┘
KNOWN_BINDER_CHAIN = "E"   # chain ID of the known binder contact fragment
KNOWN_TARGET_CHAIN = "B"   # chain ID of the target in the known PDB

# ── Validation ────────────────────────────────────────────────────────────────
VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

if TARGET_SEQUENCE:
    TARGET_SEQUENCE = TARGET_SEQUENCE.strip().upper()
    bad = set(TARGET_SEQUENCE) - VALID_AA
    if bad:
        raise ValueError(
            f"TARGET_SEQUENCE contains invalid characters: {bad}\n"
            "Use single-letter amino acid codes only."
        )
    print(f"✓ Target sequence : {TARGET_SEQUENCE[:60]}{'...' if len(TARGET_SEQUENCE)>60 else ''}")
    print(f"  Length           : {len(TARGET_SEQUENCE)} residues")
else:
    print("⚠️  TARGET_SEQUENCE is empty — will attempt auto-extraction from PDB.")
    print("   Provide the sequence explicitly above if possible.")

print(f"\n✓ Known binder chain : {KNOWN_BINDER_CHAIN}")
print("✓ Contact residues   : all residues present in the uploaded binder chain")


✓ Target sequence : MAAAMDVDTPSGTNSGAGKKRFEVKKWNAVALWAWDIVVDNCAICRNHIMDLCIECQANQ...
  Length           : 108 residues

✓ Known binder chain : E
✓ Contact residues   : all residues present in the uploaded binder chain


## Cell 4 — Upload input files

Upload:
1. **Designs CSV** — columns `name` and `sequence`
2. **Known binder PDB or CIF** — binder chain pre-trimmed to
contact residues only. May optionally contain the target chain.

Ensure Cell 3 is filled in before running this cell.


In [5]:
from google.colab import files as colab_files
import json, csv

print("Upload your designs CSV (columns: name, sequence):")
up1 = colab_files.upload()
DESIGNS_CSV = f"/content/{list(up1.keys())[0]}"
with open(DESIGNS_CSV, "wb") as f:
    f.write(list(up1.values())[0])
print(f"  ✓ Designs : {DESIGNS_CSV}")

print("\nUpload your known binder PDB/CIF (contact residues only):")
up2 = colab_files.upload()
KNOWN_PDB = f"/content/{list(up2.keys())[0]}"
with open(KNOWN_PDB, "wb") as f:
    f.write(list(up2.values())[0])
print(f"  ✓ Known binder : {KNOWN_PDB}")

print("\n✓ Files uploaded — proceed to Cell 5")


Upload your designs CSV (columns: name, sequence):


Saving Model Test - Sheet1 (1).csv to Model Test - Sheet1 (1).csv
  ✓ Designs : /content/Model Test - Sheet1 (1).csv

Upload your known binder PDB/CIF (contact residues only):


Saving testrbx1.cif to testrbx1.cif
  ✓ Known binder : /content/testrbx1.cif

✓ Files uploaded — proceed to Cell 5


In [6]:
# @title Cell 4d — FoldSeek: search target structure for co-complex binders
# Submits the uploaded TARGET PDB to FoldSeek, finds PDB entries where a
# structurally similar target appears in complex with a protein/peptide binder,
# trims each binder to its contact residues only, writes a complex PDB
# (binder as KNOWN_BINDER_CHAIN, target as KNOWN_TARGET_CHAIN) so that
# Cell 8 USalign multimer alignment works correctly, and adds them to
# all_templates for use in Cell 8 alignment.
#
# Requires: Cell 1 (USalign, gemmi), Cell 4 (KNOWN_PDB uploaded),
#           Cell 3 (TARGET_SEQUENCE, KNOWN_TARGET_CHAIN, KNOWN_BINDER_CHAIN set)
#
# Produces: all_templates — list of complex PDBs ready for Cell 8

import requests, time, os, subprocess, re
import numpy as np
import gemmi

# ── Adjustable parameters ─────────────────────────────────────────────────────
TARGET_FOLDSEEK_HITS   = 10
CONTACT_DIST_THRESHOLD = 4.0
MIN_INTERFACE_CONTACTS = 6
MIN_BINDER_RESIDUES    = 5
TARGET_FOLDSEEK_DB     = "pdb100"

_STANDARD_AA = {
    "ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY",
    "HIS","ILE","LEU","LYS","MET","PHE","PRO","SER",
    "THR","TRP","TYR","VAL"
}

TARGET_TEMPLATES_DIR = "/content/pipeline_output/target_foldseek_templates"
os.makedirs(TARGET_TEMPLATES_DIR, exist_ok=True)

print("=== Cell 4d: FoldSeek target structure search ===\n")
print(f"  Output chain IDs: binder -> '{KNOWN_BINDER_CHAIN}'  target -> '{KNOWN_TARGET_CHAIN}'")
print(f"  (matching KNOWN_PDB convention from Cell 3)\n")


# ── Step 1: extract target chain from KNOWN_PDB as FoldSeek query ─────────────

target_query_pdb = os.path.join(TARGET_TEMPLATES_DIR, "target_query.pdb")

st = gemmi.read_structure(str(KNOWN_PDB))
new_st    = gemmi.Structure()
new_st.name = "target_query"
new_model = gemmi.Model("1")
found_target_chain = False

for model in st:
    for chain in model:
        if chain.name == KNOWN_TARGET_CHAIN:
            new_model.add_chain(chain.clone())
            found_target_chain = True
    break

if not found_target_chain:
    raise ValueError(
        f"KNOWN_TARGET_CHAIN '{KNOWN_TARGET_CHAIN}' not found in {KNOWN_PDB}. "
        "Check Cell 3 settings."
    )

new_st.add_model(new_model)
new_st.write_pdb(target_query_pdb)
print(f"✓ Target chain {KNOWN_TARGET_CHAIN} extracted -> {target_query_pdb}")


# ── Step 2: submit to FoldSeek ────────────────────────────────────────────────

print(f"\nSubmitting target structure to FoldSeek ({TARGET_FOLDSEEK_DB})...")

with open(target_query_pdb, "rb") as fh:
    resp = requests.post(
        "https://search.foldseek.com/api/ticket",
        files={"q": fh},
        data={"mode": "3diaa", "database[]": TARGET_FOLDSEEK_DB},
        timeout=30,
    )
if resp.status_code != 200:
    raise RuntimeError(
        f"FoldSeek submission failed: {resp.status_code}\n{resp.text}")

ticket = resp.json()["id"]
print(f"  ticket: {ticket}", end=" ", flush=True)

for _ in range(60):
    time.sleep(3)
    status = requests.get(
        f"https://search.foldseek.com/api/ticket/{ticket}", timeout=10
    ).json()
    if status["status"] == "COMPLETE":
        break
    print(".", end="", flush=True)
else:
    raise TimeoutError("FoldSeek timed out after 3 minutes")

results  = requests.get(
    f"https://search.foldseek.com/api/result/{ticket}/0", timeout=30
).json()
raw_hits = results.get("results", [])
print(f"\n  {len(raw_hits)} db result(s) returned")


# ── Parse FoldSeek alignments ─────────────────────────────────────────────────
# FoldSeek format: results -> [{db, alignments: [[{query, target, ...}]]}]
# target field: "4f52-assembly1.cif.gz_B Description..."

def parse_foldseek_target(target_str):
    """Extract (pdb_id, chain_id) from FoldSeek target string."""
    clean  = target_str.split(" ")[0]          # strip description
    match  = re.search(r'_([A-Za-z0-9])$', clean)
    chain  = match.group(1) if match else None
    pdb_id = clean[:4].upper()
    if len(pdb_id) < 4 or not pdb_id.isalnum():
        return None, None
    return pdb_id, chain

pdb_hits  = []
seen_pdbs = set()

for db_result in raw_hits:
    for alignment_list in db_result.get("alignments", []):
        for aln in alignment_list:
            pdb_id, fs_target_chain = parse_foldseek_target(aln.get("target", ""))
            if not pdb_id:
                continue
            if pdb_id not in seen_pdbs:
                seen_pdbs.add(pdb_id)
                pdb_hits.append({
                    "pdb":          pdb_id,
                    "target_chain": fs_target_chain,
                    "prob":         aln.get("prob", aln.get("score", 0.0)),
                })
            if len(pdb_hits) >= TARGET_FOLDSEEK_HITS:
                break
        if len(pdb_hits) >= TARGET_FOLDSEEK_HITS:
            break

print(f"  {len(pdb_hits)} unique PDB entries to inspect")
for h in pdb_hits:
    print(f"    {h['pdb']} target_chain={h['target_chain']} prob={h['prob']:.3f}")
print()


# ── Helpers ───────────────────────────────────────────────────────────────────

def count_interface_contacts(st, binder_chain_name, target_chain_name):
    """
    Residue-level contact count: a binder residue is counted if ANY of its
    heavy atoms are within CONTACT_DIST_THRESHOLD Angstrom of ANY target heavy atom.
    Returns (n_contact_residues, set_of_contact_seqnums).
    """
    binder_by_res = {}
    target_atoms  = []

    for model in st:
        for chain in model:
            if chain.name == binder_chain_name:
                for res in chain:
                    coords = [
                        np.array([a.pos.x, a.pos.y, a.pos.z], dtype=np.float32)
                        for a in res if a.element.name != "H"
                    ]
                    if coords:
                        binder_by_res[res.seqid.num] = coords
            elif chain.name == target_chain_name:
                for res in chain:
                    for a in res:
                        if a.element.name != "H":
                            target_atoms.append(
                                np.array([a.pos.x, a.pos.y, a.pos.z],
                                         dtype=np.float32))
        break

    if not binder_by_res or not target_atoms:
        return 0, set()

    target_arr  = np.array(target_atoms)
    contact_res = set()

    for seqnum, coords in binder_by_res.items():
        binder_arr = np.array(coords)
        diffs      = binder_arr[:, None, :] - target_arr[None, :, :]
        min_dist   = np.sqrt((diffs ** 2).sum(axis=2)).min()
        if min_dist <= CONTACT_DIST_THRESHOLD:
            contact_res.add(seqnum)

    return len(contact_res), contact_res


def write_complex_pdb(st, binder_chain_name, contact_seqnums,
                      target_chain_name, out_path):
    """
    Write a complex PDB containing:
      - Binder contact residues as chain KNOWN_BINDER_CHAIN
      - Full target chain as chain KNOWN_TARGET_CHAIN
    This ensures Cell 8 USalign multimer alignment (-mm 1) has both chains
    and can anchor the superposition correctly via the target chain.
    Chain IDs are renamed to match KNOWN_PDB convention from Cell 3.
    """
    new_st    = gemmi.Structure()
    new_st.name = os.path.basename(out_path)[:20]
    new_model = gemmi.Model("1")

    # ── Binder: contact residues only, renamed to KNOWN_BINDER_CHAIN ──────────
    new_binder_chain = gemmi.Chain(KNOWN_BINDER_CHAIN)
    for model in st:
        for chain in model:
            if chain.name != binder_chain_name:
                continue
            for res in chain:
                if res.seqid.num in contact_seqnums:
                    new_binder_chain.add_residue(res.clone())
        break
    new_model.add_chain(new_binder_chain)

    # ── Target: full chain, renamed to KNOWN_TARGET_CHAIN ────────────────────
    new_target_chain = gemmi.Chain(KNOWN_TARGET_CHAIN)
    for model in st:
        for chain in model:
            if chain.name != target_chain_name:
                continue
            for res in chain:
                new_target_chain.add_residue(res.clone())
        break
    new_model.add_chain(new_target_chain)

    new_st.add_model(new_model)
    new_st.write_pdb(out_path)


# ── Step 3: process each PDB hit ─────────────────────────────────────────────

all_templates = []

# Known binder is always template 0
all_templates.append({
    "pdb":          "known",
    "chain":        KNOWN_BINDER_CHAIN,
    "cif_path":     str(KNOWN_PDB),
    "tmscore":      1.0,
    "source":       "known_binder",
    "bsa":          0.0,
    "mean_bfactor": 999.0,
})
print(f"[0] known_binder  (always template 0)\n")

n_added = 0

for hit in pdb_hits:
    pdb_id       = hit["pdb"]
    target_chain = hit["target_chain"]
    print(f"  {pdb_id}:", end=" ", flush=True)

    # Download CIF
    cif_path = os.path.join(TARGET_TEMPLATES_DIR, f"{pdb_id}.cif")
    if not os.path.exists(cif_path):
        r = requests.get(
            f"https://files.rcsb.org/download/{pdb_id}.cif", timeout=30)
        if r.status_code != 200:
            print("download failed")
            continue
        with open(cif_path, "w") as fh:
            fh.write(r.text)

    try:
        st = gemmi.read_structure(cif_path)
    except Exception as e:
        print(f"parse failed ({e})")
        continue

    # Collect protein/peptide chains using AA name check
    chain_names = []
    for model in st:
        for chain in model:
            aa_count = sum(1 for res in chain if res.name in _STANDARD_AA)
            if aa_count >= MIN_BINDER_RESIDUES:
                chain_names.append((chain.name, aa_count))
        break

    if not chain_names:
        print("no protein chains")
        continue

    # Verify FoldSeek's target chain exists in this CIF
    if target_chain not in [c for c, _ in chain_names]:
        print(f"target chain {target_chain} not found in CIF — skipping")
        continue

    binder_chains = [(c, n) for c, n in chain_names if c != target_chain]

    print(f"{len(chain_names)} chain(s)  target={target_chain}  "
          f"candidates={[c for c, _ in binder_chains]}")

    if not binder_chains:
        print("    no binder candidates")
        continue

    # Contact filter, trim, and write complex
    for chain_name, aa_count in binder_chains:
        n_contact_res, contact_seqnums = count_interface_contacts(
            st, chain_name, target_chain)

        print(f"    {chain_name}: {aa_count} aa  "
              f"contact_residues={n_contact_res}", end=" ")

        if n_contact_res < MIN_INTERFACE_CONTACTS:
            print(f"— FAIL (< {MIN_INTERFACE_CONTACTS})")
            continue

        # Write complex PDB: binder contacts as KNOWN_BINDER_CHAIN,
        # full target as KNOWN_TARGET_CHAIN
        out_pdb = os.path.join(
            TARGET_TEMPLATES_DIR,
            f"{pdb_id}_{chain_name}_complex.pdb")
        write_complex_pdb(st, chain_name, contact_seqnums, target_chain, out_pdb)

        all_templates.append({
            "pdb":           pdb_id,
            "chain":         KNOWN_BINDER_CHAIN,   # renamed in output PDB
            "cif_path":      out_pdb,
            "tmscore":       hit["prob"],
            "source":        f"target_foldseek_{pdb_id}_{chain_name}",
            "bsa":           0.0,
            "mean_bfactor":  999.0,
            "n_contacts":    n_contact_res,
            "n_contact_res": n_contact_res,
        })
        n_added += 1
        print(f"— ✓ ({n_contact_res} contact res, "
              f"written as {KNOWN_BINDER_CHAIN}+{KNOWN_TARGET_CHAIN})")


# ── Summary ───────────────────────────────────────────────────────────────────

print(f"\n✓ Cell 4d complete: {n_added} additional template(s) found")
print(f"  all_templates total: {len(all_templates)}\n")
for i, t in enumerate(all_templates):
    nc = f"  ({t['n_contact_res']} contact res)" if "n_contact_res" in t else ""
    print(f"  [{i}] {t['source']}{nc}")

=== Cell 4d: FoldSeek target structure search ===

  Output chain IDs: binder -> 'E'  target -> 'B'
  (matching KNOWN_PDB convention from Cell 3)

✓ Target chain B extracted -> /content/pipeline_output/target_foldseek_templates/target_query.pdb

Submitting target structure to FoldSeek (pdb100)...
  ticket: OJiEql2gh7i3G552-NOIeIk8T8QimoiKTcoDtw 
  1 db result(s) returned
  10 unique PDB entries to inspect
    4F52 target_chain=B prob=1.000
    7Z8V target_chain=R prob=1.000
    7Z8B target_chain=R prob=1.000
    3DPL target_chain=R prob=1.000
    4P5O target_chain=B prob=1.000
    3RTR target_chain=D prob=1.000
    8JAS target_chain=V prob=1.000
    3DQV target_chain=Y prob=1.000
    1LDJ target_chain=B prob=1.000
    6TTU target_chain=R prob=1.000

[0] known_binder  (always template 0)

  4F52: 6 chain(s)  target=B  candidates=['A', 'C', 'D', 'E', 'F']
    A: 258 aa  contact_residues=42 — ✓ (42 contact res, written as E+B)
    C: 254 aa  contact_residues=0 — FAIL (< 6)
    D: 85 aa  c

## Cell 6 — Stage 1: Validate inputs


In [7]:
print("Cell 6 starting...")
os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR created")

if not TARGET_SEQUENCE:
    target_chain_id = auto_detect_target_chain(KNOWN_PDB, KNOWN_BINDER_CHAIN)
    if target_chain_id:
        TARGET_SEQUENCE = get_chain_sequence(KNOWN_PDB, target_chain_id)
        print(f"  Auto-extracted target from chain {target_chain_id}: "
              f"{TARGET_SEQUENCE[:50]}... ({len(TARGET_SEQUENCE)} aa)")
    else:
        raise ValueError("Could not auto-detect target chain. Set TARGET_SEQUENCE manually in Cell 3.")
else:
    TARGET_SEQUENCE = TARGET_SEQUENCE.strip().upper()
    print(f"  Target sequence: {TARGET_SEQUENCE[:50]}... ({len(TARGET_SEQUENCE)} aa)")

print("Target sequence resolved")

known_contact_residues = get_chain_residues(KNOWN_PDB, KNOWN_BINDER_CHAIN)
print(f"  Known contact residues loaded: {len(known_contact_residues)}")
assert known_contact_residues, \
    f"Chain '{KNOWN_BINDER_CHAIN}' not found or empty in {KNOWN_PDB}."

known_contact_seq = "".join(r["aa"] for r in known_contact_residues)
print(f"  Contact fragment sequence: {known_contact_seq}")

print("Loading designs CSV...")
designs = load_designs(DESIGNS_CSV)
print(f"  Loaded {len(designs)} valid design sequences")

print("\n✓ Stage 1 complete — inputs validated")

Cell 6 starting...
OUT_DIR created
  Target sequence: MAAAMDVDTPSGTNSGAGKKRFEVKKWNAVALWAWDIVVDNCAICRNHIM... (108 aa)
Target sequence resolved
  Known contact residues loaded: 16
  Contact fragment sequence: QKDDMANRYIKFDSVR
Loading designs CSV...
  Detected no header in CSV
  Loaded 3 valid design sequences

✓ Stage 1 complete — inputs validated


## Cell 7 — Stage 2: Initial Boltz-2 screen

Runs Boltz-2 on every design sequence and filters by IPSAE.
Designs that fail the IPSAE threshold are dropped before alignment.


In [8]:
stage2_dir = os.path.join(OUT_DIR, "stage2_initial")
os.makedirs(stage2_dir, exist_ok=True)

print(f"Running initial Boltz-2 screen on {len(designs)} designs")
print(f"  recycling={RECYCLING_STEPS}, samples={DIFFUSION_SAMPLES}, "
      f"sampling_steps={SAMPLING_STEPS}")
print(f"  IPSAE screen threshold : ≥ {MIN_IPSAE_SCREEN}\n")

initial_results = []

for row in designs:
    name = row["name"]
    seq  = row["sequence"]
    print(f"  {name[:50]} ({len(seq)} aa) ...", end=" ", flush=True)
    t0   = time.time()
    work = os.path.join(stage2_dir, name)

    try:
        pred_dir   = run_boltz(name, seq, TARGET_SEQUENCE, work)
        metrics    = score_prediction(pred_dir, binder_len=len(seq),
                                      target_len=len(TARGET_SEQUENCE))
        cross_pae  = get_cross_pae_per_binder_residue(
                         pred_dir, len(seq), len(TARGET_SEQUENCE))
        cif_path   = get_best_cif(pred_dir)
        elapsed    = time.time() - t0

        passes = (metrics["ipsae_bt"] >= MIN_IPSAE_SCREEN or
                  metrics["ipsae_tb"] >= MIN_IPSAE_SCREEN)

        print(f"score={metrics['ranking_score']:.4f}  "
              f"ipTM={metrics['iptm']:.3f}  "
              f"IPSAE_bt={metrics['ipsae_bt']:.3f}  "
              f"IPSAE_tb={metrics['ipsae_tb']:.3f}  "
              f"({'PASS' if passes else 'FAIL'})  ({elapsed:.0f}s)\n"
              f"    seq: {seq}")

        if passes:
            initial_results.append({
                "design": row, "pred_dir": pred_dir,
                "metrics": metrics, "cross_pae": cross_pae,
                "structure_path": cif_path, "seq": seq, "name": name,
            })
    except Exception as e:
        print(f"FAILED: {e}")

print(f"\n✓ Stage 2 complete: {len(initial_results)}/{len(designs)} passed screen")


Running initial Boltz-2 screen on 3 designs
  recycling=3, samples=6, sampling_steps=200
  IPSAE screen threshold : ≥ 0.6

  binder_1 (80 aa) ... score=-1.7219  ipTM=0.934  IPSAE_bt=0.736  IPSAE_tb=0.839  (PASS)  (196s)
    seq: YAAEARLARLEKALEVVEKKLEEAEKELEELEEEAKKKGLPEPPDSPDLELRMAWAQYEYLKDWLEELEEKLAEAEAAVA
  binder_2 (80 aa) ... score=-1.6946  ipTM=0.929  IPSAE_bt=0.695  IPSAE_tb=0.836  (PASS)  (81s)
    seq: YVEEKIKEAEKKIEEWEKLLEELEEESKEAGVPSAADPNADISGLSEEKRAIAETLKFGYELNKEGIEKLEEKIEELKKL
  binder_3 (80 aa) ... score=-1.6777  ipTM=0.917  IPSAE_bt=0.732  IPSAE_tb=0.789  (PASS)  (80s)
    seq: SAEARAAGEAAVKELLESGAVSEEEAKDAEPVVEWLELYADDPEDRELGLNALQLAVDNPERAAELLEQAGAYARTVFGV

✓ Stage 2 complete: 3/3 passed screen


## Cell 8 — Stage 3: USalign complex alignment

Aligns each predicted binder+target complex against the known
binder+target complex using USalign multimer mode (-mm 1).
Both chains are used jointly in the superposition, so the target
chain anchors the relative orientation and the binder alignment
reflects true binding geometry rather than a free superposition.
The residue map is extracted from the binder chain rows only.


In [9]:
# @title Cell 8 — Stage 3: USalign complex alignment (all templates)
# Aligns each Boltz-2 predicted complex against every template in all_templates.
# For each design position, keeps the candidate with the lowest per-residue RMSD
# across all templates. Known binder gets a spatial proximity bonus — a FoldSeek
# template must beat it by KNOWN_BINDER_RMSD_BONUS Å to displace it.
# Stores per_design_candidates on each result for Cell 9.

import gemmi as _gemmi

stage3_dir = Path(OUT_DIR) / "stage3_alignment"
stage3_dir.mkdir(parents=True, exist_ok=True)

BOLTZ_BINDER_CHAIN      = "A"
BOLTZ_TARGET_CHAIN      = "B"
KNOWN_BINDER_RMSD_BONUS = 0.2   # Å — FoldSeek must beat known binder by this margin

# Fallback if Cell 4d was not run
if "all_templates" not in dir() or not all_templates:
    print("[warn] all_templates not found — using KNOWN_PDB only")
    all_templates = [{
        "pdb":      "known",
        "chain":    KNOWN_BINDER_CHAIN,
        "cif_path": str(KNOWN_PDB),
        "tmscore":  1.0,
        "source":   "known_binder",
    }]

print(f"Aligning {len(initial_results)} predicted complexes "
      f"against {len(all_templates)} template(s)...")
print(f"  USalign mode         : -mm 1 (multimer, complex-vs-complex)")
print(f"  Query                : chain {BOLTZ_BINDER_CHAIN} (binder) + {BOLTZ_TARGET_CHAIN} (target)")
print(f"  Reference            : chain {KNOWN_BINDER_CHAIN} (binder) + {KNOWN_TARGET_CHAIN} (target)")
print(f"  RMSD gate            : ≤ {MAX_RMSD_SCREEN} Å")
print(f"  Known binder bonus   : {KNOWN_BINDER_RMSD_BONUS} Å (FoldSeek must beat by this margin)\n")

aligned_results = []


def _degap(s):
    return s.replace("-", "")


def _parse_tmscore_rmsd(stdout):
    """Parse TM-score and RMSD from USalign verbose stdout."""
    tmscore = rmsd = 0.0
    for line in stdout.splitlines():
        s = line.strip()
        if s.startswith("TM-score=") and "Structure_1" in s:
            try:
                tmscore = float(s.split("TM-score=")[1].split()[0])
            except Exception:
                pass
        if s.startswith("Aligned length="):
            try:
                rmsd = float(s.split("RMSD=")[1].split(",")[0])
            except Exception:
                pass
    return tmscore, rmsd


def _parse_residue_map(stdout, binder_len_query, contact_len_ref):
    """
    Parse binder residue map from USalign stdout alignment block.
    Returns {query_0based: ref_0based}.
    """
    residue_map  = {}
    stdout_lines = stdout.splitlines()
    query_full = ref_full = None

    for i, line in enumerate(stdout_lines):
        if line.strip().startswith("Aligned length="):
            aln_idx = i + 6
            if aln_idx + 2 < len(stdout_lines):
                query_full = stdout_lines[aln_idx].strip()
                ref_full   = stdout_lines[aln_idx + 2].strip()
            break

    if not query_full or not ref_full:
        return residue_map

    query_parts = [p for p in query_full.split("*") if p]
    ref_parts   = [p for p in ref_full.split("*")   if p]

    q_idx = next((j for j, p in enumerate(query_parts)
                  if len(_degap(p)) == binder_len_query), None)
    r_idx = next((j for j, p in enumerate(ref_parts)
                  if len(_degap(p)) == contact_len_ref), None)

    if q_idx is not None and r_idx is not None:
        qs = query_parts[q_idx]
        rs = ref_parts[r_idx]
        if len(qs) == len(rs):
            qi = ki = 0
            for qc, kc in zip(qs, rs):
                if qc != "-" and kc != "-":
                    residue_map[qi] = ki
                if qc != "-": qi += 1
                if kc != "-": ki += 1

    return residue_map


for res in initial_results:
    name     = res["name"]
    cif_path = res["structure_path"]
    if not cif_path or not os.path.exists(cif_path):
        print(f"  {name}: no structure file — skipping")
        continue

    work = stage3_dir / name
    work.mkdir(parents=True, exist_ok=True)

    # Convert Boltz-2 CIF to PDB once — reused for all templates
    tmp_pdb = str(work / f"{name}_query.pdb")
    st = _gemmi.read_structure(str(cif_path))
    st.write_pdb(tmp_pdb)

    per_design_candidates = {}
    best_tmscore     = 0.0
    best_rmsd        = float("inf")
    best_residue_map = {}
    best_superposed  = tmp_pdb

    for tmpl in all_templates:
        tmpl_pdb    = tmpl["cif_path"]
        tmpl_source = tmpl["source"]
        ref_chain   = tmpl["chain"]
        is_known    = (tmpl_source == "known_binder")

        # Load contact residues for this template
        try:
            tmpl_contact_residues = get_chain_residues(tmpl_pdb, ref_chain)
        except Exception as e:
            print(f"  [warn] could not load template {tmpl_source}: {e}")
            continue
        if not tmpl_contact_residues:
            print(f"  [warn] no residues in chain {ref_chain} of {tmpl_source}")
            continue

        prefix = str(work / f"{name}_aln_{tmpl_source.replace('/', '_')[:30]}")
        cmd = [
            USALIGN_BIN,
            tmp_pdb, tmpl_pdb,
            "-mm",  "1",
            "-ter", "0",
            "-o",   prefix,
        ]
        try:
            proc = subprocess.run(
                cmd, capture_output=True, text=True, timeout=180)
            if proc.returncode != 0:
                print(f"  [warn] USalign failed for {tmpl_source}: "
                      f"{proc.stderr[-100:]}")
                continue
        except Exception as e:
            print(f"  [warn] USalign error for {tmpl_source}: {e}")
            continue

        tmscore, rmsd = _parse_tmscore_rmsd(proc.stdout)

        if rmsd > MAX_RMSD_SCREEN:
            print(f"    [{tmpl_source}] TM={tmscore:.3f} "
                  f"RMSD={rmsd:.2f} Å — FAIL")
            continue

        residue_map = _parse_residue_map(
            proc.stdout, len(res["seq"]), len(tmpl_contact_residues))

        superposed_pdb = prefix + ".pdb"
        if not os.path.exists(superposed_pdb):
            superposed_pdb = tmp_pdb

        print(f"    [{tmpl_source}] TM={tmscore:.3f}  "
              f"RMSD={rmsd:.2f} Å  mapped={len(residue_map)}"
              + ("  [known]" if is_known else ""))

        # Track best global alignment
        if rmsd < best_rmsd:
            best_rmsd        = rmsd
            best_tmscore     = tmscore
            best_residue_map = residue_map
            best_superposed  = superposed_pdb

        # Load superposed design residues for per-residue RMSD
        try:
            design_residues = get_chain_residues(superposed_pdb, BOLTZ_BINDER_CHAIN)
        except Exception:
            design_residues = []

        if design_residues:
            for d_idx, k_idx in residue_map.items():
                if d_idx >= len(design_residues):
                    continue
                if k_idx >= len(tmpl_contact_residues):
                    continue

                rmsd_res = per_residue_rmsd(
                    design_residues[d_idx], tmpl_contact_residues[k_idx])

                existing = per_design_candidates.get(d_idx)

                if existing is None:
                    # No candidate yet — take this one
                    per_design_candidates[d_idx] = {
                        "design_idx":             d_idx,
                        "known_aa":               tmpl_contact_residues[k_idx]["aa"],
                        "_rmsd_ref":              rmsd_res,
                        "_template_source":       tmpl_source,
                        "_tmpl_contact_residues": tmpl_contact_residues,
                        "_superposed_pdb":        superposed_pdb,
                    }
                else:
                    # Position already has a candidate — apply bonus logic
                    existing_is_known = (existing["_template_source"] == "known_binder")

                    if existing_is_known:
                        # Current holder is known binder — FoldSeek must beat by bonus
                        effective_rmsd = rmsd_res - KNOWN_BINDER_RMSD_BONUS
                    else:
                        # Current holder is FoldSeek — known binder always wins ties,
                        # other FoldSeek templates compete on raw RMSD
                        if is_known:
                            effective_rmsd = rmsd_res - KNOWN_BINDER_RMSD_BONUS
                        else:
                            effective_rmsd = rmsd_res

                    if effective_rmsd < existing["_rmsd_ref"]:
                        per_design_candidates[d_idx] = {
                            "design_idx":             d_idx,
                            "known_aa":               tmpl_contact_residues[k_idx]["aa"],
                            "_rmsd_ref":              rmsd_res,
                            "_template_source":       tmpl_source,
                            "_tmpl_contact_residues": tmpl_contact_residues,
                            "_superposed_pdb":        superposed_pdb,
                        }

    if best_rmsd == float("inf"):
        print(f"  {name}: no template passed RMSD gate — skipping")
        continue

    # Summary: show how many positions each template source contributed
    source_counts = {}
    for cinfo in per_design_candidates.values():
        src = cinfo["_template_source"]
        source_counts[src] = source_counts.get(src, 0) + 1

    print(f"  {name[:50]}: best RMSD={best_rmsd:.2f} Å  "
          f"positions_from_all_templates={len(per_design_candidates)}")
    for src, count in sorted(source_counts.items(), key=lambda x: -x[1]):
        print(f"    {src}: {count} position(s)")

    res["tmscore"]               = best_tmscore
    res["rmsd_global"]           = best_rmsd
    res["residue_map"]           = best_residue_map
    res["superposed_pdb"]        = best_superposed
    res["per_design_candidates"] = per_design_candidates
    aligned_results.append(res)

print(f"\n✓ Stage 3 complete: {len(aligned_results)}/{len(initial_results)} "
      f"passed RMSD gate  ({len(all_templates)} templates used)")

Aligning 3 predicted complexes against 22 template(s)...
  USalign mode         : -mm 1 (multimer, complex-vs-complex)
  Query                : chain A (binder) + B (target)
  Reference            : chain E (binder) + B (target)
  RMSD gate            : ≤ 3.0 Å
  Known binder bonus   : 0.2 Å (FoldSeek must beat by this margin)

    [known_binder] TM=0.364  RMSD=2.00 Å  mapped=8  [known]
    [target_foldseek_4F52_A] TM=0.342  RMSD=2.57 Å  mapped=5
    [target_foldseek_4F52_E] TM=0.363  RMSD=2.10 Å  mapped=8
    [target_foldseek_7Z8V_C] TM=0.388  RMSD=2.57 Å  mapped=1
    [target_foldseek_7Z8V_D] TM=0.382  RMSD=2.30 Å  mapped=0
    [target_foldseek_7Z8B_C] TM=0.345  RMSD=2.20 Å  mapped=1
    [target_foldseek_3DPL_C] TM=0.406  RMSD=2.00 Å  mapped=0
    [target_foldseek_4P5O_A] TM=0.332  RMSD=1.57 Å  mapped=0
    [target_foldseek_4P5O_I] TM=0.366  RMSD=2.13 Å  mapped=10
    [target_foldseek_4P5O_K] TM=0.341  RMSD=1.82 Å  mapped=3
    [target_foldseek_3RTR_A] TM=0.365  RMSD=2.27 Å  mapped=1

In [10]:
# @title Cell 9 — Stage 4: Residue flagging (cross-chain PAE + RMSD + beta-sheet + SASA)
# Consumes per_design_candidates from Cell 8: each position already carries
# its best-template reference (lowest per-residue RMSD across all templates).
# Design residues are loaded from that candidate's superposed PDB so
# coordinates are in the correct reference frame.

import sys

stage4_dir = Path(OUT_DIR) / "stage4_flagging"
stage4_dir.mkdir(parents=True, exist_ok=True)

print(f"Flagging substitution candidates...")
print(f"  cross-chain PAE ≤ {MAX_CROSS_PAE} Å | RMSD ≤ {RMSD_THRESHOLD} Å | "
      f"SASA ≥ {MIN_SASA} Å² | dot_product ≥ {DOT_PRODUCT_MIN}\n")

flagged_results = []

for res in aligned_results:
    name        = res["name"]
    seq         = res["seq"]
    cif_path    = res["structure_path"]
    cross_pae_r = res["cross_pae"]

    # per_design_candidates: design_idx -> {known_aa, _rmsd_ref,
    #   _template_source, _tmpl_contact_residues, _superposed_pdb}
    pdc = res.get("per_design_candidates", {})

    print(f"  {name[:50]}:", end=" ", flush=True)

    if not pdc:
        print("no candidates from alignment — skipping")
        continue

    # DSSP from original CIF (frame-independent)
    dssp_map = get_dssp(cif_path, "A")

    # Cache loaded design_residues per superposed PDB to avoid redundant reads
    _superposed_cache = {}

    # SASA from best superposed PDB (representative for this design)
    best_superposed = res.get("superposed_pdb", cif_path)
    sasa_map = get_sasa(best_superposed, "A")

    candidates = []

    for design_idx, cinfo in pdc.items():
        superposed_pdb = cinfo["_superposed_pdb"]
        tmpl_source    = cinfo["_template_source"]
        known_aa       = cinfo["known_aa"]
        rmsd_ref       = cinfo["_rmsd_ref"]   # pre-computed in Cell 8

        # Load design residues from this candidate's superposed PDB
        if superposed_pdb not in _superposed_cache:
            try:
                _superposed_cache[superposed_pdb] = get_chain_residues(
                    superposed_pdb, "A")
            except Exception as e:
                _superposed_cache[superposed_pdb] = []
        design_residues = _superposed_cache[superposed_pdb]

        if not design_residues or design_idx >= len(design_residues):
            continue

        d_res  = design_residues[design_idx]
        des_aa = d_res["aa"]

        # Skip proline substitutions and identity positions
        if known_aa in EXCLUDE_AA or known_aa == des_aa:
            continue

        # Gate 1: cross-chain PAE
        res_cross_pae = (cross_pae_r[design_idx]
                         if design_idx < len(cross_pae_r)
                         else float("inf"))
        if res_cross_pae > MAX_CROSS_PAE:
            continue

        # Gate 2: per-residue RMSD (pre-computed against best template)
        if rmsd_ref > RMSD_THRESHOLD:
            continue

        # Gate 3: beta-sheet sidechain orientation
        ss  = dssp_map.get(design_idx, "C")
        dot = float("nan")
        if ss == "E" and dssp_map:
            if 0 < design_idx < len(design_residues) - 1:
                normal = sheet_normal(
                    design_residues[design_idx - 1]["ca"],
                    d_res["ca"],
                    design_residues[design_idx + 1]["ca"],
                )
                dot = cb_dot(d_res["ca"], d_res["cb"], normal)
                if not math.isnan(dot) and dot < DOT_PRODUCT_MIN:
                    continue

        # Gate 4: solvent-accessible surface area
        sasa = sasa_map.get(design_idx, 0.0)
        if sasa < MIN_SASA:
            continue

        candidates.append({
            "design_idx":      design_idx,
            "design_aa":       des_aa,
            "known_aa":        known_aa,
            "rmsd":            round(rmsd_ref, 3),
            "cross_pae":       round(res_cross_pae, 2),
            "sasa":            round(sasa, 2),
            "dot_product":     round(dot, 3) if not math.isnan(dot) else None,
            "ss":              ss,
            "template_source": tmpl_source,
            "trunk_score":     0.0,
        })

    # Sort by RMSD ascending — best geometric match first
    candidates.sort(key=lambda c: c["rmsd"])

    print(f"{len(candidates)} candidates (from {len(pdc)} aligned positions)",
          end="", flush=True)

    # ── Trunk scoring ──────────────────────────────────────────────────────────
    TRUNK_SCRIPT = "/content/trunk_score.py"
    if candidates and os.path.exists(TRUNK_SCRIPT):
        import tempfile, subprocess as _sp
        tmp_out = tempfile.NamedTemporaryFile(suffix=".json", delete=False)
        tmp_out.close()
        try:
            _proc = _sp.run([
                sys.executable, TRUNK_SCRIPT,
                "--pred_dir",   res["pred_dir"],
                "--binder_seq", seq,
                "--binder_len", str(len(seq)),
                "--target_len", str(len(TARGET_SEQUENCE)),
                "--candidates", json.dumps(candidates),
                "--out",        tmp_out.name,
            ], capture_output=True, text=True, timeout=180)

            if _proc.returncode == 0:
                trunk_result = json.load(open(tmp_out.name))
                trunk_scores = trunk_result["scores"]
                method       = trunk_result["method"]
                for c in candidates:
                    c["trunk_score"] = trunk_scores.get(
                        str(c["design_idx"]), 0.0)
                candidates.sort(key=lambda c: c["trunk_score"], reverse=True)
                print(f"  trunk:{method}", end="")
            else:
                print(f"  trunk:failed", end="")
                if _proc.stderr:
                    print(f" ({_proc.stderr.strip()[-100:]})", end="")
        except Exception as _e:
            print(f"  trunk:error({_e})", end="")
        finally:
            try: os.unlink(tmp_out.name)
            except: pass

    print()  # newline after status line

    with open(stage4_dir / f"{name}_candidates.json", "w") as f:
        json.dump({"name": name, "candidates": candidates}, f, indent=2)

    if candidates:
        res["candidates"] = candidates
        flagged_results.append(res)

total_cands = sum(len(r.get("candidates", [])) for r in flagged_results)
print(f"\n✓ Stage 4 complete: {total_cands} total candidates "
      f"across {len(flagged_results)} designs")

Flagging substitution candidates...
  cross-chain PAE ≤ 10.0 Å | RMSD ≤ 3.0 Å | SASA ≥ 5.0 Å² | dot_product ≥ 0.1

  binder_1: 4 candidates (from 38 aligned positions)  trunk:pair_emb_norm
  binder_2: 4 candidates (from 42 aligned positions)  trunk:pair_emb_norm
  binder_3: 4 candidates (from 31 aligned positions)  trunk:pair_emb_norm

✓ Stage 4 complete: 12 total candidates across 3 designs


## Cell 10 — Stage 5: Variant generation

Generates combinatorial substitution variants for each design.
The unmodified original sequence is always included as a baseline.


In [11]:
stage5_dir = Path(OUT_DIR) / "stage5_variants"
stage5_dir.mkdir(parents=True, exist_ok=True)

all_variants = {}

for res in flagged_results:
    name     = res["name"]
    seq      = res["seq"]
    cands    = res["candidates"]
    variants = []

    # Original as baseline (always included)
    variants.append({
        "name": f"{name}_original", "sequence": seq,
        "substitutions": [], "is_original": True, "parent": name,
    })

    # Combinatorial substitutions in order of increasing substitution count
    for n_subs in range(1, min(MAX_SUBSTITUTIONS, len(cands)) + 1):
        for combo in itertools.combinations(cands, n_subs):
            if len(variants) >= MAX_VARIANTS + 1:
                break
            seq_list = list(seq)
            subs     = []
            for c in combo:
                seq_list[c["design_idx"]] = c["known_aa"]
                subs.append({"idx": c["design_idx"],
                              "from_aa": c["design_aa"],
                              "to_aa":   c["known_aa"]})
            new_seq = "".join(seq_list)
            if new_seq == seq:
                continue
            sub_str  = "_".join(f"{s['idx']}{s['from_aa']}{s['to_aa']}"
                                 for s in subs)
            variants.append({
                "name":          f"{name}_sub_{sub_str}",
                "sequence":      new_seq,
                "substitutions": subs,
                "is_original":   False,
                "parent":        name,
            })
        if len(variants) >= MAX_VARIANTS + 1:
            break

    all_variants[name] = variants
    with open(stage5_dir / f"{name}_variants.json", "w") as f:
        json.dump(variants, f, indent=2)
    print(f"  {name}: {len(variants) - 1} variants + 1 original")

total_var = sum(len(v) for v in all_variants.values())
print(f"\n✓ Stage 5 complete: {total_var} total variants "
      f"across {len(all_variants)} designs")


  binder_1: 9 variants + 1 original
  binder_2: 9 variants + 1 original
  binder_3: 9 variants + 1 original

✓ Stage 5 complete: 30 total variants across 3 designs


## Cell 11 — Stage 6: Variant Boltz-2 re-prediction

Predicts every variant sequence against the full target.
Serial execution — one prediction at a time to respect GPU memory.
No structural input is carried forward (sequence-only prediction).


In [12]:
stage6_dir = Path(OUT_DIR) / "stage6_variant_preds"
stage6_dir.mkdir(parents=True, exist_ok=True)

variant_predictions = {}

for design_name, variants in all_variants.items():
    preds = []
    print(f"\n  [{design_name}] Predicting {len(variants)} variants...")

    for i, var in enumerate(variants):
        tag = "original" if var["is_original"] else f"variant {i}"
        print(f"    {tag}: {var['name'][:60]} ...", end=" ", flush=True)
        t0   = time.time()
        work = str(stage6_dir / design_name / var["name"])

        try:
            pred_dir = run_boltz(var["name"], var["sequence"],
                                 TARGET_SEQUENCE, work)
            metrics  = score_prediction(pred_dir,
                                        binder_len=len(var["sequence"]),
                                        target_len=len(TARGET_SEQUENCE))
            cif_path = get_best_cif(pred_dir)
            elapsed  = time.time() - t0

            print(f"score={metrics['ranking_score']:.4f}  "
                  f"ipTM={metrics['iptm']:.3f}  "
                  f"IPSAE_bt={metrics['ipsae_bt']:.3f}  ({elapsed:.0f}s)")

            preds.append({
                "variant": var, "pred_dir": pred_dir, "metrics": metrics,
                "cif_path": cif_path,
            })
        except Exception as e:
            print(f"FAILED: {e}")

    variant_predictions[design_name] = preds

print(f"\n✓ Stage 6 complete")



  [binder_1] Predicting 10 variants...
    original: binder_1_original ... score=-1.6857  ipTM=0.922  IPSAE_bt=0.720  (80s)
    variant 1: binder_1_sub_63WH ... score=-1.7215  ipTM=0.935  IPSAE_bt=0.727  (80s)
    variant 2: binder_1_sub_61KE ... score=-1.7076  ipTM=0.926  IPSAE_bt=0.742  (80s)
    variant 3: binder_1_sub_58ES ... score=-1.7125  ipTM=0.936  IPSAE_bt=0.705  (81s)
    variant 4: binder_1_sub_43PF ... score=-1.6860  ipTM=0.925  IPSAE_bt=0.700  (80s)
    variant 5: binder_1_sub_63WH_61KE ... score=-1.7115  ipTM=0.928  IPSAE_bt=0.739  (81s)
    variant 6: binder_1_sub_63WH_58ES ... score=-1.6963  ipTM=0.928  IPSAE_bt=0.709  (80s)
    variant 7: binder_1_sub_63WH_43PF ... score=-1.7205  ipTM=0.930  IPSAE_bt=0.750  (80s)
    variant 8: binder_1_sub_61KE_58ES ... score=-1.7059  ipTM=0.933  IPSAE_bt=0.705  (81s)
    variant 9: binder_1_sub_61KE_43PF ... score=-1.6353  ipTM=0.907  IPSAE_bt=0.697  (80s)

  [binder_2] Predicting 10 variants...
    original: binder_2_original ... 

## Cell 12 — Stage 7: MOSAIC scoring + RMSD gate

Scores all variant predictions with the MOSAIC ranking formula.
Variants whose predicted structure diverges from the original
parent prediction by more than RMSD_GATE_VS_PARENT Å are excluded.


In [13]:
# @title Cell 12 — Stage 7: MOSAIC scoring + RMSD gate

import gemmi as _gemmi_s7
import tempfile as _tempfile_s7

stage7_dir = Path(OUT_DIR) / "stage7_scoring"
stage7_dir.mkdir(parents=True, exist_ok=True)

print(f"Computing MOSAIC ranking scores and applying RMSD gate "
      f"(≤ {RMSD_GATE_VS_PARENT} Å vs parent)...\n")

all_scored = {}

for design_name, preds in variant_predictions.items():
    if not preds:
        continue

    parent_pred = next((p for p in preds if p["variant"]["is_original"]), None)
    scored      = []

    for p in preds:
        metrics     = p["metrics"]
        rmsd_parent = 0.0
        passed_gate = True

        if parent_pred and not p["variant"]["is_original"]:
            if p["cif_path"] and parent_pred["cif_path"]:
                tmp1_path = tmp2_path = None
                try:
                    # Convert CIFs to PDB for USalign compatibility
                    with _tempfile_s7.NamedTemporaryFile(
                            suffix=".pdb", delete=False) as t1:
                        tmp1_path = t1.name
                    with _tempfile_s7.NamedTemporaryFile(
                            suffix=".pdb", delete=False) as t2:
                        tmp2_path = t2.name

                    _gemmi_s7.read_structure(
                        str(p["cif_path"])).write_pdb(tmp1_path)
                    _gemmi_s7.read_structure(
                        str(parent_pred["cif_path"])).write_pdb(tmp2_path)

                    # Single-chain RMSD — binder chain A vs binder chain A
                    # No -o flag so full output goes to stdout
                    cmd = [
                        USALIGN_BIN,
                        tmp1_path, tmp2_path,
                        "-chain2", "A",
                        "-ter",    "0",
                    ]
                    proc = subprocess.run(
                        cmd, capture_output=True, text=True, timeout=120)

                    if proc.returncode != 0:
                        raise RuntimeError(
                            f"USalign failed:\n{proc.stderr}")

                    # Parse RMSD from "Aligned length= X, RMSD= Y, ..." line
                    for line in proc.stdout.splitlines():
                        if "RMSD=" in line and "Aligned length=" in line:
                            try:
                                rmsd_parent = float(
                                    line.strip()
                                        .split("RMSD=")[1]
                                        .split(",")[0]
                                        .strip()
                                )
                                break
                            except Exception:
                                pass

                    if rmsd_parent == 0.0:
                        print(f"\n  [debug] could not parse RMSD for "
                              f"{p['variant']['name']} "
                              f"({len(proc.stdout.splitlines())} stdout lines)")
                        for dbg in proc.stdout.splitlines()[:6]:
                            print(f"    {repr(dbg)}")

                    passed_gate = rmsd_parent <= RMSD_GATE_VS_PARENT

                except Exception as e:
                    print(f"  [warn] RMSD-vs-parent failed for "
                          f"{p['variant']['name']}: {e}")
                finally:
                    for tmp in [tmp1_path, tmp2_path]:
                        if tmp:
                            try: os.unlink(tmp)
                            except OSError: pass

        scored.append({
            "design":           design_name,
            "variant_name":     p["variant"]["name"],
            "is_original":      p["variant"]["is_original"],
            "sequence":         p["variant"]["sequence"],
            "substitutions":    p["variant"]["substitutions"],
            "ranking_score":    metrics["ranking_score"],
            "iptm":             metrics["iptm"],
            "ptm":              metrics["ptm"],
            "plddt":            metrics["plddt"],
            "ipsae_bt":         metrics["ipsae_bt"],
            "ipsae_tb":         metrics["ipsae_tb"],
            "rmsd_vs_parent":   round(rmsd_parent, 3),
            "passed_rmsd_gate": passed_gate,
            "structure_path":   str(p["cif_path"]) if p["cif_path"] else "",
        })

    all_scored[design_name] = scored
    with open(stage7_dir / f"{design_name}_scores.json", "w") as f:
        json.dump(scored, f, indent=2)

print("✓ Stage 7 complete")

Computing MOSAIC ranking scores and applying RMSD gate (≤ 1.5 Å vs parent)...

✓ Stage 7 complete


## Cell 13 — Stage 8: Final ranking and CSV output


In [14]:
# @title Cell 13 — Stage 8: Final ranking and CSV output

stage8_dir = Path(OUT_DIR) / "stage8_ranking"
stage8_dir.mkdir(parents=True, exist_ok=True)

all_rows = []

# ── Grafted variants ───────────────────────────────────────────────────────────
for design_name, scored in all_scored.items():
    passing  = [s for s in scored if s["passed_rmsd_gate"]]
    excluded = len(scored) - len(passing)

    orig       = next((s for s in passing if s["is_original"]), None)
    orig_score = orig["ranking_score"] if orig else 0.0

    for s in passing:
        delta = s["ranking_score"] - orig_score
        subs  = "; ".join(
            f"{sub['idx']+1}{sub['from_aa']}→{sub['to_aa']}"
            for sub in s["substitutions"]
        ) or "—"
        all_rows.append({
            "design":          s["design"],
            "variant_name":    s["variant_name"],
            "is_original":     s["is_original"],
            "substitutions":   subs,
            "sequence":        s["sequence"],
            "ranking_score":   round(s["ranking_score"], 6),
            "delta_vs_orig":   round(delta, 6),
            "iptm":            round(s["iptm"], 4),
            "ptm":             round(s["ptm"], 4),
            "plddt":           round(s["plddt"], 4),
            "ipsae_bt":        round(s["ipsae_bt"], 4),
            "ipsae_tb":        round(s["ipsae_tb"], 4),
            "rmsd_vs_parent":  s["rmsd_vs_parent"],
            "structure_path":  s["structure_path"],
            "source":          "grafted",
        })

# ── All initial screen results (including those with no grafting candidates) ───
for res in initial_results:
    name    = res["name"]
    metrics = res["metrics"]

    # Skip if already represented via grafting pipeline
    if any(r["design"] == name for r in all_rows):
        continue

    all_rows.append({
        "design":          name,
        "variant_name":    f"{name}_original",
        "is_original":     True,
        "substitutions":   "—",
        "sequence":        res["seq"],
        "ranking_score":   round(metrics["ranking_score"], 6),
        "delta_vs_orig":   0.0,
        "iptm":            round(metrics["iptm"], 4),
        "ptm":             round(metrics["ptm"], 4),
        "plddt":           round(metrics["plddt"], 4),
        "ipsae_bt":        round(metrics["ipsae_bt"], 4),
        "ipsae_tb":        round(metrics["ipsae_tb"], 4),
        "rmsd_vs_parent":  0.0,
        "structure_path":  str(res["structure_path"]) if res["structure_path"] else "",
        "source":          "initial_screen",
    })

# ── Unified ranking ────────────────────────────────────────────────────────────
all_rows.sort(key=lambda r: r["ranking_score"])
for i, row in enumerate(all_rows, 1):
    row["rank"] = i

# ── Print ──────────────────────────────────────────────────────────────────────
print("Final unified ranking  (lower ranking_score = better)\n")
print("=" * 80)
print(f"{'#':>3}  {'variant':<45}  {'score':>8}  {'ipTM':>6}  "
      f"{'bt':>6}  {'tb':>6}  {'source'}")
print("-" * 80)

for row in all_rows:
    marker = " ←" if row["is_original"] else ""
    print(f"  {row['rank']:>2}  {row['variant_name'][:45]:<45}  "
          f"{row['ranking_score']:>8.4f}  "
          f"{row['iptm']:>6.3f}  "
          f"{row['ipsae_bt']:>6.4f}  "
          f"{row['ipsae_tb']:>6.4f}  "
          f"{row['source']}{marker}")

print("=" * 80)
print(f"\n{len(all_rows)} total entries  "
      f"({sum(1 for r in all_rows if r['source']=='grafted')} grafted variants, "
      f"{sum(1 for r in all_rows if r['source']=='initial_screen')} initial screen only)\n")

# ── Write CSV ──────────────────────────────────────────────────────────────────
if not all_rows:
    print("⚠️  No variants to write.")
else:
    # Move rank to first column
    fieldnames = ["rank"] + [k for k in all_rows[0].keys() if k != "rank"]
    with open(OUTPUT_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)
    print(f"✓ Results written to {OUTPUT_CSV}  ({len(all_rows)} rows)")
    print(f"✓ Stage 8 complete — pipeline finished\n")
    colab_files.download(OUTPUT_CSV)
    print(f"✓ Downloaded {OUTPUT_CSV}")

Final unified ranking  (lower ranking_score = better)

  #  variant                                           score    ipTM      bt      tb  source
--------------------------------------------------------------------------------
   1  binder_3_original                               -1.7228   0.932  0.7446  0.8370  grafted ←
   2  binder_1_sub_63WH                               -1.7215   0.935  0.7268  0.8463  grafted
   3  binder_1_sub_63WH_43PF                          -1.7205   0.930  0.7503  0.8307  grafted
   4  binder_2_original                               -1.7166   0.932  0.7328  0.8368  grafted ←
   5  binder_2_sub_69LK                               -1.7146   0.931  0.7195  0.8469  grafted
   6  binder_1_sub_58ES                               -1.7125   0.936  0.7047  0.8484  grafted
   7  binder_3_sub_64EQ                               -1.7115   0.926  0.7546  0.8155  grafted
   8  binder_1_sub_63WH_61KE                          -1.7115   0.928  0.7387  0.8273  grafted
   9  b

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloaded binders_optimised_ranked.csv


In [15]:
#@title Cell - 14 (Optional): Substitution Checks
import json

for name in ["binder_1", "binder_2", "binder_3"]:
    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"{'='*60}")

    # Load candidates
    with open(f"/content/pipeline_output/stage4_flagging/{name}_candidates.json") as f:
        data = json.load(f)
    candidates = data["candidates"]

    # Get per_design_candidates from aligned_results
    res = next((r for r in aligned_results if r["name"] == name), None)
    if res is None:
        print("  not in aligned_results")
        continue

    pdc = res.get("per_design_candidates", {})

    # Count by template source
    source_counts = {}
    for cinfo in pdc.values():
        src = cinfo["_template_source"]
        source_counts[src] = source_counts.get(src, 0) + 1

    print(f"  Total aligned positions: {len(pdc)}")
    print(f"  By template source:")
    for src, count in sorted(source_counts.items(), key=lambda x: -x[1]):
        marker = " [known]" if src == "known_binder" else ""
        print(f"    {src}: {count}{marker}")

    print(f"\n  Passed all gates: {len(candidates)}")
    for c in candidates:
        # Find the template source for this position
        cinfo = pdc.get(c["design_idx"], {})
        src   = cinfo.get("_template_source", "unknown")
        rmsd_raw = cinfo.get("_rmsd_ref", 0.0)
        marker = " [known]" if src == "known_binder" else " [foldseek]"
        print(f"    pos={c['design_idx']:3d}  {c['design_aa']}->{c['known_aa']}  "
              f"rmsd={rmsd_raw:.3f}  pae={c['cross_pae']:.2f}  "
              f"sasa={c['sasa']:.2f}  trunk={c['trunk_score']:.4f}{marker}")


binder_1
  Total aligned positions: 38
  By template source:
    target_foldseek_3RTR_A: 9
    target_foldseek_6TTU_D: 6
    target_foldseek_8JAS_B: 4
    target_foldseek_3RTR_B: 4
    target_foldseek_4P5O_I: 3
    known_binder: 2 [known]
    target_foldseek_4F52_E: 2
    target_foldseek_4F52_A: 2
    target_foldseek_6TTU_U: 2
    target_foldseek_4P5O_K: 2
    target_foldseek_1LDJ_A: 1
    target_foldseek_3RTR_F: 1

  Passed all gates: 4
    pos= 63  W->H  rmsd=2.410  pae=4.73  sasa=8.81  trunk=0.6579 [foldseek]
    pos= 61  K->E  rmsd=2.787  pae=4.87  sasa=95.13  trunk=0.4013 [foldseek]
    pos= 58  E->S  rmsd=1.798  pae=4.66  sasa=23.89  trunk=0.3020 [foldseek]
    pos= 43  P->F  rmsd=1.659  pae=6.43  sasa=30.08  trunk=0.2484 [known]

binder_2
  Total aligned positions: 42
  By template source:
    target_foldseek_4P5O_I: 11
    target_foldseek_8JAS_B: 6
    target_foldseek_3RTR_A: 5
    target_foldseek_4F52_E: 4
    target_foldseek_6TTU_D: 4
    target_foldseek_3RTR_C: 3
    known_